In [1]:
# Cell 1: Install dependencies
!pip install -q datasets pandas matplotlib

In [2]:
# Cell 2: Load dataset
from datasets import load_dataset
import pandas as pd
import numpy as np
import re

dataset = load_dataset("arbml/ashaar")
dataset

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:112: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/4.71k [00:00<?, ?B/s]

dataset_infos.json:   0%|          | 0.00/34.7k [00:00<?, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/126M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/151M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/254630 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url', 'poet name', 'poet description', 'poet url', 'poet era', 'poet location', 'poem description', 'poem language type'],
        num_rows: 254630
    })
})

In [3]:
# Cell 3: Convert to pandas
# Check available splits first
print(dataset.keys())

# Usually use train if available
split_name = list(dataset.keys())[0]
df = dataset[split_name].to_pandas()

print(df.shape)
df.head()

dict_keys(['train'])
(254630, 12)


,poem title,poem meter,poem verses,poem theme,poem url,poet name,poet description,poet url,poet era,poet location,poem description,poem language type
0,أصبح الملك للذي فطر الخلق,بحر الخفيف,"[أَصبَحَ المُلك لِلَّذي فَطر الخَل, قَ بِتَقدي...",قصيدة دينية,https://www.aldiwan.net/poem16182.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
1,من أي مولى ارتجي,بحر مجزوء الرمل,"[مِن أَي مَولى اِرتَجي, وَلاي باب التَجي, وَال...",قصيدة دينية,https://www.aldiwan.net/poem16183.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
2,العبد عبدك يا من أنت سيده,بحر البسيط,"[العَبد عَبدك يا مَن أَنتَ سَيدهُ, وَلَيسَ غَي...",قصيدة ذم,https://www.aldiwan.net/poem16184.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
3,لو كنت أطمع بالمنام توهما,بحر الكامل,"[لَو كُنتَ أَطمَع بِالمَنام تَوهما, لَسالَت طَ...",قصيدة عامه,https://www.aldiwan.net/poem16185.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None
4,يعد علي أنفاسي ذنوبا,بحر الوافر,"[يعد عَليَّ أَنفاسي ذُنوباً, إِذا ما قُلت أَفد...",قصيدة عامه,https://www.aldiwan.net/poem16186.html,الامير منجك باشا,منجك بن محمد بن منجك بن ابي بكر بن عبد القادر ...,https://www.aldiwan.net/cat-poet-alamir-mnczyk...,العصر العثماني,None,None,None


In [4]:
# Cell 4: Basic column inspection
df.columns

Index(['poem title', 'poem meter', 'poem verses', 'poem theme', 'poem url',
       'poet name', 'poet description', 'poet url', 'poet era',
       'poet location', 'poem description', 'poem language type'],
      dtype='object')

In [5]:
# Cell 5: Total number of unique meters in the full corpus
meter_col = "poem meter"

unique_meters = df[meter_col].dropna().unique()
num_unique_meters = len(unique_meters)

print("Number of unique meters:", num_unique_meters)
print("\nMeters:")
for m in sorted(unique_meters):
    print(m)

Number of unique meters: 101

Meters:
البسيط
التفعيله
الحداء
الخفيف
الدوبيت
الرجز
الرمل
السريع
السلسلة
الصخري
الطويل
الكامل
الكان كان
اللويحاني
المتدارك
المتقارب
المجتث
المديد
المسحوب
المضارع
المقتضب
المنسرح
المواليا
الموشح
الهجيني
الهزج
الوافر
بحر أحذ الكامل
بحر أحذ المديد
بحر أحذ الوافر
بحر البسيط
بحر التفعيله
بحر الخبب
بحر الخفيف
بحر الدوبيت
بحر الرجز
بحر الرمل
بحر السريع
بحر السلسلة
بحر الطويل
بحر القوما
بحر الكامل
بحر الكامل المقطوع
بحر المتدارك
بحر المتدارك المنهوك
بحر المتقارب
بحر المجتث
بحر المديد
بحر المضارع
بحر المقتضب
بحر المنسرح
بحر المواليا
بحر الهزج
بحر الوافر
بحر تفعيلة الرجز
بحر تفعيلة الرمل
بحر تفعيلة الكامل
بحر تفعيلة المتقارب
بحر مجزوء البسيط
بحر مجزوء الخفيف
بحر مجزوء الدوبيت
بحر مجزوء الرجز
بحر مجزوء الرمل
بحر مجزوء الرمل 
بحر مجزوء السريع
بحر مجزوء الطويل
بحر مجزوء الكامل
بحر مجزوء المتدارك
بحر مجزوء المتقارب
بحر مجزوء المجتث
بحر مجزوء المديد
بحر مجزوء المنسرح
بحر مجزوء المواليا
بحر مجزوء الهزج
بحر مجزوء الوافر
بحر مجزوء موشح
بحر مخلع البسيط
بحر مخلع الرجز
بحر مخل

In [6]:
# Cell 6: Meter distribution
meter_distribution = (
    df[meter_col]
    .fillna("MISSING")
    .value_counts()
    .reset_index()
)

meter_distribution.columns = ["meter", "poem_count"]
meter_distribution["percentage"] = 100 * meter_distribution["poem_count"] / len(df)

meter_distribution

,meter,poem_count,percentage
0,MISSING,101277,39.774182
1,الطويل,19746,7.754781
2,الكامل,16151,6.342929
3,بحر الطويل,14823,5.821388
4,البسيط,12198,4.790480
...,...,...,...
97,الحداء,1,0.000393
98,مجزوء الخفيف,1,0.000393
99,عدة أبحر,1,0.000393
100,بسيط,1,0.000393


In [7]:
# Cell 7: The 20 meters with the fewest poems
least_20_meters = meter_distribution.sort_values("poem_count", ascending=True).head(20)
least_20_meters

,meter,poem_count,percentage
80,بحر تفعيلة الرجز,1,0.000393
81,بحر تفعيلة المتقارب,1,0.000393
82,بحر مجزوء الرمل,1,0.000393
83,بحر القوما,1,0.000393
84,بحر الخبب,1,0.000393
85,بحر مجزوء المنسرح,1,0.000393
86,بحر مجزوء المواليا,1,0.000393
101,اللويحاني,1,0.000393
96,الصخري,1,0.000393
97,الحداء,1,0.000393


In [8]:
# Cell 8: Number of poems with at least one empty hemistich
verses_col = "poem verses"

def has_empty_hemistich(verses):
    if not isinstance(verses, (list, np.ndarray)):
        return True
    return any((v is None) or (str(v).strip() == "") for v in verses)

df["has_empty_hemistich"] = df[verses_col].apply(has_empty_hemistich)

num_empty_hemistich = df["has_empty_hemistich"].sum()
perc_empty_hemistich = 100 * num_empty_hemistich / len(df)

print("Poems with at least one empty hemistich:", num_empty_hemistich)
print(f"Percentage: {perc_empty_hemistich:.2f}%")

Poems with at least one empty hemistich: 431
Percentage: 0.17%


In [9]:
# Cell 9: Theme categories and distribution
theme_col = "poem theme"

theme_distribution = (
    df[theme_col]
    .fillna("MISSING")
    .value_counts()
    .reset_index()
)

theme_distribution.columns = ["theme", "poem_count"]
theme_distribution["percentage"] = 100 * theme_distribution["poem_count"] / len(df)

print("Number of theme categories:", df[theme_col].fillna("MISSING").nunique())
theme_distribution

Number of theme categories: 19


,theme,poem_count,percentage
0,MISSING,187110,73.483093
1,قصيدة قصيره,25911,10.175942
2,قصيدة عامه,20611,8.094490
3,قصيدة مدح,5165,2.028433
4,قصيدة رومنسيه,4074,1.599969
5,قصيدة حزينه,2277,0.894239
6,قصيدة عتاب,2032,0.798021
7,قصيدة هجاء,1614,0.633861
8,قصيدة غزل,1416,0.556101
9,قصيدة دينية,1186,0.465774


In [10]:
# Cell 10: Exact list of theme categories
themes = sorted(df[theme_col].fillna("MISSING").unique())

for t in themes:
    print(t)

MISSING
قصيدة اعتذار
قصيدة الاناشيد
قصيدة المعلقات
قصيدة حزينه
قصيدة دينية
قصيدة ذم
قصيدة رثاء
قصيدة رومنسيه
قصيدة سياسية
قصيدة شوق
قصيدة عامه
قصيدة عتاب
قصيدة غزل
قصيدة فراق
قصيدة قصيره
قصيدة مدح
قصيدة هجاء
قصيدة وطنيه


In [11]:
# Cell 11: Era distribution
era_col = "poet era"

era_distribution = (
    df[era_col]
    .fillna("MISSING")
    .value_counts()
    .reset_index()
)

era_distribution.columns = ["era", "poem_count"]
era_distribution["percentage"] = 100 * era_distribution["poem_count"] / len(df)

era_distribution

,era,poem_count,percentage
0,MISSING,107209,42.103837
1,العصر الحديث,55137,21.653772
2,العصر العباسي,30413,11.943997
3,العصر المملوكي,19059,7.484978
4,العصر العثماني,11872,4.662451
5,المغرب والأندلس,6614,2.597494
6,العصر الفاطمي,4939,1.939677
7,العصر الأندلسي,4770,1.873306
8,العصر الأموي,3958,1.554412
9,العصر الأيوبي,3125,1.227271


In [12]:
# Cell 12: Poem language type distribution
language_col = "poem language type"

language_distribution = (
    df[language_col]
    .fillna("MISSING")
    .value_counts()
    .reset_index()
)

language_distribution.columns = ["language_type", "poem_count"]
language_distribution["percentage"] = 100 * language_distribution["poem_count"] / len(df)

language_distribution

,language_type,poem_count,percentage
0,فصيح,153722,60.370734
1,MISSING,71223,27.971174
2,فصحى,20852,8.189137
3,عامي,8503,3.339355
4,شعبي,298,0.117033
5,-,32,0.012567


In [13]:
# Cell 13: Empty values in poem description
description_col = "poem description"

def is_empty_description(x):
    if x is None:
        return True
    if isinstance(x, float) and pd.isna(x):
        return True
    if isinstance(x, list):
        return len(x) == 0 or all(str(i).strip() == "" for i in x if i is not None)
    return str(x).strip() == ""

df["empty_description"] = df[description_col].apply(is_empty_description)

num_empty_desc = df["empty_description"].sum()
perc_empty_desc = 100 * num_empty_desc / len(df)

print("Empty poem descriptions:", num_empty_desc)
print(f"Percentage: {perc_empty_desc:.2f}%")

Empty poem descriptions: 240160
Percentage: 94.32%


In [14]:
# Cell 14: Hemistich count per poem
def count_hemistichs(verses):
    if not isinstance(verses, (list, np.ndarray)):
        return 0
    return len(verses)

df["num_hemistichs"] = df[verses_col].apply(count_hemistichs)

hemistich_stats = df["num_hemistichs"].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])
hemistich_stats

,num_hemistichs
count,254630.000000
mean,30.298307
std,67.717473
min,0.000000
25%,6.000000
50%,14.000000
75%,36.000000
90%,71.000000
95%,100.000000
99%,198.000000


In [15]:
# Cell 15: Hemistich count distribution
hemistich_distribution = (
    df["num_hemistichs"]
    .value_counts()
    .sort_index()
    .reset_index()
)

hemistich_distribution.columns = ["num_hemistichs", "poem_count"]
hemistich_distribution["percentage"] = 100 * hemistich_distribution["poem_count"] / len(df)

hemistich_distribution

,num_hemistichs,poem_count,percentage
0,0,1,0.000393
1,1,325,0.127636
2,2,8827,3.466599
3,3,601,0.236029
4,4,42506,16.693241
...,...,...,...
663,3601,1,0.000393
664,3603,2,0.000785
665,4734,2,0.000785
666,6982,1,0.000393


In [16]:
# Cell 16: Group hemistich counts into useful ranges
bins = [-1, 0, 2, 4, 10, 20, 40, 100, np.inf]
labels = ["0", "1-2", "3-4", "5-10", "11-20", "21-40", "41-100", ">100"]

df["hemistich_range"] = pd.cut(df["num_hemistichs"], bins=bins, labels=labels)

hemistich_range_distribution = (
    df["hemistich_range"]
    .value_counts()
    .sort_index()
    .reset_index()
)

hemistich_range_distribution.columns = ["hemistich_range", "poem_count"]
hemistich_range_distribution["percentage"] = 100 * hemistich_range_distribution["poem_count"] / len(df)

hemistich_range_distribution

,hemistich_range,poem_count,percentage
0,0,1,0.000393
1,1-2,9152,3.594235
2,3-4,43107,16.929270
3,5-10,54769,21.509249
4,11-20,45591,17.904803
5,21-40,46816,18.385893
6,41-100,42634,16.743510
7,>100,12560,4.932647


In [24]:
# Cell 17: Poems exceeding 40 hemistichs
num_over_40_hemistichs = (df["num_hemistichs"] > 40).sum()
perc_over_40_hemistichs = 100 * num_over_40_hemistichs / len(df)

print("Poems with more than 40 hemistichs:", num_over_20_hemistichs)
print(f"Percentage: {perc_over_40_hemistichs:.2f}%")

Poems with more than 40 hemistichs: 55194
Percentage: 21.68%


In [18]:
# Cell 18: Optional artifact checks: Latin, digits, URLs
def join_verses(verses):
    if not isinstance(verses, (list, np.ndarray)):
        return ""
    return " ".join(str(v) for v in verses if v is not None)

df["poem_text_joined"] = df[verses_col].apply(join_verses)

df["has_latin"] = df["poem_text_joined"].str.contains(r"[A-Za-z]", regex=True, na=False)
df["has_digits"] = df["poem_text_joined"].str.contains(r"\d", regex=True, na=False)
df["has_url"] = df["poem_text_joined"].str.contains(r"http[s]?://|www\.", regex=True, na=False)
df["has_tatweel"] = df["poem_text_joined"].str.contains("ـ", regex=False, na=False)

artifact_stats = pd.DataFrame({
    "artifact": ["Latin characters", "Digits", "URLs", "Tatweel"],
    "poems_affected": [
        df["has_latin"].sum(),
        df["has_digits"].sum(),
        df["has_url"].sum(),
        df["has_tatweel"].sum()
    ]
})

artifact_stats["percentage"] = 100 * artifact_stats["poems_affected"] / len(df)
artifact_stats

,artifact,poems_affected,percentage
0,Latin characters,505,0.198327
1,Digits,5953,2.337902
2,URLs,48,0.018851
3,Tatweel,115717,45.445156


In [19]:
# Cell 19: Create a compact summary table
summary = pd.DataFrame({
    "Statistic": [
        "Total poems",
        "Unique meters",
        "Theme categories",
        "Poems with at least one empty hemistich",
        "Empty poem descriptions",
        "Mean hemistich count",
        "Median hemistich count",
        "Maximum hemistich count",
        "Poems with >20 hemistichs"
    ],
    "Value": [
        len(df),
        num_unique_meters,
        df[theme_col].fillna("MISSING").nunique(),
        int(num_empty_hemistich),
        int(num_empty_desc),
        round(df["num_hemistichs"].mean(), 2),
        int(df["num_hemistichs"].median()),
        int(df["num_hemistichs"].max()),
        int(num_over_20_hemistichs)
    ]
})

summary

,Statistic,Value
0,Total poems,254630.0
1,Unique meters,101.0
2,Theme categories,19.0
3,Poems with at least one empty hemistich,431.0
4,Empty poem descriptions,240160.0
5,Mean hemistich count,30.3
6,Median hemistich count,14.0
7,Maximum hemistich count,11608.0
8,Poems with >20 hemistichs,102010.0


In [20]:
# Cell 20: Save all statistics as CSV files
output_dir = "/content/ashaar_preprocessing_stats"

import os
os.makedirs(output_dir, exist_ok=True)

summary.to_csv(f"{output_dir}/summary.csv", index=False)
meter_distribution.to_csv(f"{output_dir}/meter_distribution.csv", index=False)
least_20_meters.to_csv(f"{output_dir}/least_20_meters.csv", index=False)
theme_distribution.to_csv(f"{output_dir}/theme_distribution.csv", index=False)
era_distribution.to_csv(f"{output_dir}/era_distribution.csv", index=False)
language_distribution.to_csv(f"{output_dir}/language_type_distribution.csv", index=False)
hemistich_distribution.to_csv(f"{output_dir}/hemistich_distribution.csv", index=False)
hemistich_range_distribution.to_csv(f"{output_dir}/hemistich_range_distribution.csv", index=False)
artifact_stats.to_csv(f"{output_dir}/artifact_stats.csv", index=False)

print("Saved CSV files to:", output_dir)

Saved CSV files to: /content/ashaar_preprocessing_stats


In [21]:
# Cell 21: Download outputs as a zip
!zip -r /content/ashaar_preprocessing_stats.zip /content/ashaar_preprocessing_stats

  adding: content/ashaar_preprocessing_stats/ (stored 0%)
  adding: content/ashaar_preprocessing_stats/summary.csv (deflated 38%)
  adding: content/ashaar_preprocessing_stats/era_distribution.csv (deflated 45%)
  adding: content/ashaar_preprocessing_stats/language_type_distribution.csv (deflated 19%)
  adding: content/ashaar_preprocessing_stats/theme_distribution.csv (deflated 47%)
  adding: content/ashaar_preprocessing_stats/least_20_meters.csv (deflated 73%)
  adding: content/ashaar_preprocessing_stats/artifact_stats.csv (deflated 18%)
  adding: content/ashaar_preprocessing_stats/meter_distribution.csv (deflated 68%)
  adding: content/ashaar_preprocessing_stats/hemistich_range_distribution.csv (deflated 33%)
  adding: content/ashaar_preprocessing_stats/hemistich_distribution.csv (deflated 77%)
